# VAL-01: 데이터 로딩 & 전처리 검증


In [7]:
import pandas as pd
import numpy as np
import math
import warnings
warnings.filterwarnings('ignore')

print('라이브러리 로딩 완료')

라이브러리 로딩 완료


---
## 0. 헬퍼 함수 (load_structured_data.py 그대로)

In [8]:
def read_csv_flexible(path, **kwargs):
   
    for encoding in ["utf-8-sig", "cp949", "euc-kr", "utf-8"]:
        try:
            return pd.read_csv(path, encoding=encoding, **kwargs)
        except Exception:
            pass
    raise RuntimeError(f"CSV 읽기 실패: {path}")

def clean_numeric(series: pd.Series) -> pd.Series:
    
    return pd.to_numeric(
        series.astype(str)
              .str.replace(',', '', regex=False)
              .str.replace('-', '0', regex=False)
              .str.strip(),
        errors='coerce'
    )

def safe_ratio(numerator, denominator, digits=6):
    
    if denominator in [0, None] or pd.isna(denominator):
        return None
    value = numerator / denominator
    if math.isinf(value) or math.isnan(value):
        return None
    return round(value, digits)

print('헬퍼 함수 정의 완료')

헬퍼 함수 정의 완료


---
## 1. 가스 데이터 (process_gas_data 동일 로직)

In [9]:
gas_raw = read_csv_flexible('../data/gas.csv', skiprows=3, header=None)

gas_df = gas_raw[[1, 3, 12, 21, 30, 39, 48]].copy()
gas_df.columns = ['district', '2019', '2020', '2021', '2022', '2023', '2024']

gas_df['district'] = gas_df['district'].astype(str).str.strip()
gas_df = gas_df[~gas_df['district'].isin(['소계', '동별(2)', 'nan'])].copy()

gas_melted = gas_df.melt(
    id_vars=['district'], var_name='year', value_name='gas_supply'
)
gas_melted['year'] = gas_melted['year'].astype(int)
gas_melted['gas_supply'] = clean_numeric(gas_melted['gas_supply'])
gas_melted = gas_melted.dropna(subset=['gas_supply']).copy()
gas_melted['gas_supply'] = gas_melted['gas_supply'].astype(int)

print(f'가스 데이터: {gas_melted.shape}')
print(f'  자치구 수: {gas_melted.district.nunique()}')
print(f'  자치구 목록: {sorted(gas_melted.district.unique())}')

가스 데이터: (150, 3)
  자치구 수: 25
  자치구 목록: ['강남구', '강동구', '강북구', '강서구', '관악구', '광진구', '구로구', '금천구', '노원구', '도봉구', '동대문구', '동작구', '마포구', '서대문구', '서초구', '성동구', '성북구', '송파구', '양천구', '영등포구', '용산구', '은평구', '종로구', '중구', '중랑구']


---
## 2. 인구 데이터 (process_pop_data 동일 로직)

In [10]:
pop_raw = read_csv_flexible('../data/pop.csv')
pop_raw['year']    = (pop_raw['기준_년분기_코드'] // 10).astype(int)
pop_raw['quarter'] = (pop_raw['기준_년분기_코드'] % 10).astype(int)

pop_filtered = pop_raw[
    (pop_raw['year'] >= 2019) &
    (pop_raw['year'] <= 2024) &
    (pop_raw['quarter'] == 4)
].copy()

pop_df = pop_filtered[['year', '자치구_코드_명', '총_상주인구_수', '총_가구_수']].copy()
pop_df.columns = ['year', 'district', 'total_resident_population', 'total_households']

pop_df['district'] = pop_df['district'].astype(str).str.strip()
pop_df['total_resident_population'] = clean_numeric(pop_df['total_resident_population'])
pop_df['total_households']          = clean_numeric(pop_df['total_households'])
pop_df = pop_df.dropna(subset=['total_resident_population', 'total_households']).copy()
pop_df['total_resident_population'] = pop_df['total_resident_population'].astype(int)
pop_df['total_households']          = pop_df['total_households'].astype(int)

print(f'인구 데이터: {pop_df.shape}')
print(f'  자치구 수: {pop_df.district.nunique()}')
print(f'  연도: {sorted(pop_df.year.unique())}')

인구 데이터: (150, 4)
  자치구 수: 25
  연도: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


---
## 3. 전력 데이터 (process_electricity_data 동일 로직)

> ETL은 `header=None` 없이 CSV 헤더를 사용하고 pandas 자동 부여 컬럼명으로 접근한다.

In [11]:
ele_raw = read_csv_flexible('../data/ele_used.csv')

ele_data = ele_raw.iloc[3:].copy()
ele_data.columns = ele_raw.columns
ele_data['district'] = ele_data['자치구별(2)'].astype(str).str.strip()
ele_data = ele_data[~ele_data['district'].isin(['소계', '합계', '자치구별(2)', 'nan'])].copy()

print('전력 CSV 컬럼 목록 (첫 30개):')
print(list(ele_raw.columns[:30]))
print()
print(f'실제 데이터 행 수: {len(ele_data)}')

전력 CSV 컬럼 목록 (첫 30개):
['자치구별(1)', '자치구별(2)', '2019', '2019.1', '2019.2', '2019.3', '2019.4', '2019.5', '2019.6', '2019.7', '2019.8', '2019.9', '2019.10', '2019.11', '2020', '2020.1', '2020.2', '2020.3', '2020.4', '2020.5', '2020.6', '2020.7', '2020.8', '2020.9', '2020.10', '2020.11', '2021', '2021.1', '2021.2', '2021.3']

실제 데이터 행 수: 25


In [12]:
year_col_map = {
    2019: {'home_usage': '2019.1', 'public_usage': '2019.2', 'service_usage': '2019.3', 'industry_usage': '2019.8'},
    2020: {'home_usage': '2020.1', 'public_usage': '2020.2', 'service_usage': '2020.3', 'industry_usage': '2020.8'},
    2021: {'home_usage': '2021.1', 'public_usage': '2021.2', 'service_usage': '2021.3', 'industry_usage': '2021.8'},
    2022: {'home_usage': '2022.1', 'public_usage': '2022.2', 'service_usage': '2022.3', 'industry_usage': '2022.8'},
    2023: {'home_usage': '2023.1', 'public_usage': '2023.2', 'service_usage': '2023.3', 'industry_usage': '2023.8'},
    2024: {'home_usage': '2024.1', 'public_usage': '2024.2', 'service_usage': '2024.3', 'industry_usage': '2024.8'},
}

ele_list = []
for year, cols in year_col_map.items():
    temp = ele_data[['district', cols['home_usage'], cols['public_usage'], cols['service_usage'], cols['industry_usage']]].copy()
    temp.columns = ['district', 'home_usage', 'public_usage', 'service_usage', 'industry_usage']

    for col in ['home_usage', 'public_usage', 'service_usage', 'industry_usage']:
        temp[col] = clean_numeric(temp[col])

    temp = temp.dropna(subset=['home_usage', 'public_usage', 'service_usage', 'industry_usage']).copy()

    for col in ['home_usage', 'public_usage', 'service_usage', 'industry_usage']:
        temp[col] = temp[col].astype(int)

    temp['year'] = year
    temp['total_usage'] = temp['home_usage'] + temp['public_usage'] + temp['service_usage'] + temp['industry_usage']
    temp['home_ratio']     = temp.apply(lambda r: safe_ratio(r['home_usage'],     r['total_usage']), axis=1)
    temp['public_ratio']   = temp.apply(lambda r: safe_ratio(r['public_usage'],   r['total_usage']), axis=1)
    temp['service_ratio']  = temp.apply(lambda r: safe_ratio(r['service_usage'],  r['total_usage']), axis=1)
    temp['industry_ratio'] = temp.apply(lambda r: safe_ratio(r['industry_usage'], r['total_usage']), axis=1)

    ele_list.append(temp[['district', 'year',
                           'home_usage', 'public_usage', 'service_usage', 'industry_usage',
                           'total_usage',
                           'home_ratio', 'public_ratio', 'service_ratio', 'industry_ratio']])

ele_df = pd.concat(ele_list, ignore_index=True)


sample = ele_df[ele_df.year == 2019].merge(
    ele_data[['district', '2019']].rename(columns={'2019': 'csv_total'}), on='district'
)
sample['csv_total'] = pd.to_numeric(sample['csv_total'].astype(str).str.replace(',', ''), errors='coerce')
diff = (sample['total_usage'] - sample['csv_total']).abs().max()

print(f'전력 데이터: {ele_df.shape}')
print(f'자치구 수: {ele_df.district.nunique()}')
print(f'연도: {sorted(ele_df.year.unique())}')
print()
print(f'합계 검증 (2019, 최대 오차): {diff:.0f}')
if diff < 100:
    print('→ ✅ total_usage ≈ CSV 합계 (industry_usage 컬럼 정확)')
else:
    print('→ ❌ 합계 불일치 — industry_usage 컬럼 재확인 필요')

전력 데이터: (150, 11)
자치구 수: 25
연도: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

합계 검증 (2019, 최대 오차): 1
→ ✅ total_usage ≈ CSV 합계 (industry_usage 컬럼 정확)


---
## 4. 통합

In [13]:
merged = pop_df.merge(gas_melted, on=['district', 'year'], how='inner')
merged = merged.merge(ele_df,     on=['district', 'year'], how='inner')

merged['gas_supply_ratio'] = merged.apply(
    lambda r: safe_ratio(r['gas_supply'], r['total_households']), axis=1
)

FEATURE_COLS = [
    'total_resident_population', 'total_households', 'gas_supply_ratio',
    'home_ratio', 'public_ratio', 'service_ratio', 'industry_ratio'
]

print(f'통합 데이터: {merged.shape}')
print()
print('결측치 확인:')
na_counts = merged[FEATURE_COLS].isnull().sum()
print(na_counts[na_counts > 0].to_string() if na_counts.any() else '  결측치 없음')

통합 데이터: (150, 15)

결측치 확인:
  결측치 없음


---
## 5. df_feat 구성 및 검증

In [14]:

df_feat = (
    merged[['year', 'district'] + FEATURE_COLS]
    .dropna()
    .sort_values(['year', 'district'])
    .reset_index(drop=True)
)

print(f'df_feat: {df_feat.shape}')
print(f'결측치: {df_feat.isnull().sum().sum()}개')
print()

year_counts = df_feat.groupby('year').size()
print('연도별 자치구 수:')
print(year_counts.to_string())
print()

n_districts = df_feat.district.nunique()
n_years     = df_feat.year.nunique()
expected    = n_districts * n_years
print(f'자치구 {n_districts}개 × 연도 {n_years}개 = 기대 {expected}행, 실제 {len(df_feat)}행')

if len(df_feat) == expected:
    print('→ ✅ 완전한 패널 데이터')
else:
    print(f'→ ⚠️ {expected - len(df_feat)}행 누락')
    full_idx = pd.MultiIndex.from_product(
        [sorted(df_feat.year.unique()), sorted(df_feat.district.unique())],
        names=['year', 'district']
    )
    actual_idx = pd.MultiIndex.from_frame(df_feat[['year', 'district']])
    missing = full_idx.difference(actual_idx)
    print(f'   누락: {missing.tolist()}')

df_feat: (150, 9)
결측치: 0개

연도별 자치구 수:
year
2019    25
2020    25
2021    25
2022    25
2023    25
2024    25

자치구 25개 × 연도 6개 = 기대 150행, 실제 150행
→ ✅ 완전한 패널 데이터


In [15]:
print('=== 피처 기술통계 ===')
print(df_feat[FEATURE_COLS].describe().round(4).to_string())

=== 피처 기술통계 ===
       total_resident_population  total_households  gas_supply_ratio  home_ratio  public_ratio  service_ratio  industry_ratio
count                   150.0000          150.0000          150.0000    150.0000      150.0000       150.0000        150.0000
mean                 380329.1733       176003.1800            1.0034      0.3278        0.0822         0.5570          0.0330
std                  125727.0938        54675.8016            0.0582      0.1076        0.0501         0.1138          0.0320
min                  120961.0000        61517.0000            0.8634      0.0809        0.0152         0.3805          0.0025
25%                  308115.0000       143436.0000            0.9712      0.2324        0.0442         0.4577          0.0111
50%                  381547.0000       180421.0000            0.9987      0.3397        0.0617         0.5535          0.0191
75%                  459137.5000       201076.2500            1.0427      0.4218        0.1080        

---
## 6. CSV 저장

In [17]:
OUT_PATH = 'data/df_feat_clean.csv'
df_feat.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

verify = pd.read_csv(OUT_PATH, encoding='utf-8-sig')
assert verify.shape == df_feat.shape, f'저장 오류: {verify.shape} ≠ {df_feat.shape}'

print(f'✅ 저장 완료: {OUT_PATH}  shape={df_feat.shape}')
print()
print('다음 단계: val_02_kmeans.ipynb 실행')

✅ 저장 완료: data/df_feat_clean.csv  shape=(150, 9)

다음 단계: val_02_kmeans.ipynb 실행
